# 01 -- Comparative Evaluation Harness Demo

Extends course 4's Ragas-style faithfulness/relevancy scoring pattern (see
`04-Capco-Model-Risk-Monitoring`) to compare two **mocked** model outputs on a synthetic
narrative-synthesis task borrowed from course 2's AML case-narrative task -- reproduced here entirely
offline with fabricated data. This is the same claim-decomposition scoring technique this course's
Chapter 2 (now retargeted at agentic, multi-agent evaluation) still relies on for narrative-quality
sub-metrics -- for instance, scoring the Underwriting Memo Drafting Agent's synthesized output against
what the upstream agents actually found, the cross-agent-consistency metric Chapter 2 describes.

Outputs are deliberately anonymized as **"Model A"** and **"Model B"** throughout the scoring pass,
mirroring the blind-evaluation design from Chapter 2 -- the mapping back to real model names is only
revealed at the very end, after scoring is complete, exactly the way a real blind review would work.

No real API keys, no network calls, no external LLM SDKs -- everything here is deterministic,
seeded, offline Python plus `numpy`/`pandas`.

In [1]:
import random
import numpy as np
import pandas as pd

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

pd.set_option("display.max_colwidth", 80)
print("Environment ready. Offline, seeded, no network calls.")

Environment ready. Offline, seeded, no network calls.


## 1. Synthetic evaluation set

Five synthetic "cases," each with a short set of source facts (standing in for retrieved KYC /
transaction / prior-case-note context) and a **ground-truth key fact** the narrative should surface
to be considered faithful. This mirrors the six-section case-narrative task from course 2, simplified
to the single claim-grounding property this demo focuses on.

In [2]:
SYNTHETIC_CASES = [
    {
        "case_id": "SYN-001",
        "context_facts": [
            "Customer opened account in 2019, stated occupation: freelance consultant.",
            "Three deposits of $9,400, $9,600, and $9,100 within a five-day window.",
            "Prior case note (2022): customer explained irregular deposits as inheritance-related, "
            "verified via estate documents.",
        ],
        "key_fact": "irregular deposits explained as inheritance, verified via estate documents",
    },
    {
        "case_id": "SYN-002",
        "context_facts": [
            "Customer opened account in 2021, stated occupation: retail store owner.",
            "Rapid movement of funds through two intermediary accounts within 48 hours.",
            "No prior case notes on file for this customer.",
        ],
        "key_fact": "no prior case history on file",
    },
    {
        "case_id": "SYN-003",
        "context_facts": [
            "Customer opened account in 2017, stated occupation: software engineer.",
            "Transaction volume roughly 40x the customer's expected monthly profile.",
            "Prior case note (2023): customer again cited inheritance-related funds, documents not "
            "requested this time.",
        ],
        "key_fact": "second inheritance claim, verification documents not requested",
    },
    {
        "case_id": "SYN-004",
        "context_facts": [
            "Customer opened account in 2020, stated occupation: import/export trader.",
            "Structuring pattern: six deposits just under the $10,000 reporting threshold.",
            "Prior case note (2021): closed as false positive, deposits matched invoice records.",
        ],
        "key_fact": "prior false-positive closure, deposits matched invoice records",
    },
    {
        "case_id": "SYN-005",
        "context_facts": [
            "Customer opened account in 2023, stated occupation: graduate student.",
            "Single large wire transfer inconsistent with stated occupation and expected profile.",
            "No prior case notes on file for this customer.",
        ],
        "key_fact": "wire transfer inconsistent with stated occupation",
    },
]

print(f"Loaded {len(SYNTHETIC_CASES)} synthetic evaluation cases.")

Loaded 5 synthetic evaluation cases.


## 2. Two mocked model clients

`MockModelA` and `MockModelB` stand in for two real LLM backends. Neither calls a real API -- both are
small, deterministic text generators built from the context facts, with **different quality
characteristics** baked in on purpose, so the scoring harness below has something real to
distinguish:

- **Model A** drops the *last* context fact roughly a third of the time (simulating a model that
  under-weights information late in a long retrieved context -- the "lost in the middle" effect
  discussed in chapter 3).
- **Model B** reliably includes every context fact.

The harness scoring these two doesn't know which is which -- see the anonymization step below.

In [3]:
class MockModelA:
    # Simulates a model that sometimes drops the last context fact when drafting.
    name = "internal-backend-alpha"

    def generate_narrative(self, context_facts):
        facts_to_use = context_facts
        # ~1/3 of the time, silently drop the last fact (often the most information-dense one
        # in these synthetic cases, by construction).
        if len(context_facts) > 1 and random.random() < 0.35:
            facts_to_use = context_facts[:-1]
        sentences = " ".join(facts_to_use)
        return f"Case Summary: {sentences}"


class MockModelB:
    # Simulates a model that reliably includes every context fact.
    name = "internal-backend-beta"

    def generate_narrative(self, context_facts):
        sentences = " ".join(context_facts)
        return f"Case Summary: {sentences}"


model_a = MockModelA()
model_b = MockModelB()
print("Two mock model clients ready:", model_a.name, "/", model_b.name)

Two mock model clients ready: internal-backend-alpha / internal-backend-beta


## 3. Ragas-style faithfulness/relevancy scoring (simplified, offline)

Course 4 covers the real Ragas framework in depth. This demo reproduces the *shape* of two of its
core metrics with simple, dependency-free heuristics suitable for an offline notebook:

- **Faithfulness** (simplified): does the generated narrative contain the case's designated
  *key fact*? A real Ragas faithfulness score decomposes the answer into claims and checks each
  against the retrieved context with an LLM judge; here we approximate that with a keyword-overlap
  check against the key fact, which is enough to demonstrate the comparison pattern without a real
  judge model.
- **Relevancy** (simplified): what fraction of the *context facts* actually appear (by keyword
  overlap) in the generated narrative -- a proxy for whether the narrative used the material it was
  given, rather than ignoring parts of it.

In [4]:
STOPWORDS = {"the", "a", "an", "of", "in", "on", "to", "for", "and", "this",
             "with", "as", "was", "is", "this's"}


def _keywords(text):
    return {w.strip(".,:;()").lower() for w in text.split()
            if w.strip(".,:;()").lower() not in STOPWORDS}


def score_faithfulness(narrative, key_fact):
    # Fraction of the key fact's keywords that appear in the generated narrative.
    key_kw = _keywords(key_fact)
    narrative_kw = _keywords(narrative)
    if not key_kw:
        return 1.0
    hit = len(key_kw & narrative_kw)
    return round(hit / len(key_kw), 3)


def score_relevancy(narrative, context_facts):
    # Fraction of context facts substantially reflected (by keyword overlap) in the narrative.
    narrative_kw = _keywords(narrative)
    covered = 0
    for fact in context_facts:
        fact_kw = _keywords(fact)
        if not fact_kw:
            continue
        overlap = len(fact_kw & narrative_kw) / len(fact_kw)
        if overlap >= 0.6:
            covered += 1
    return round(covered / len(context_facts), 3) if context_facts else 1.0


# Smoke-test the scorers on a trivial example before running the full harness.
_demo_narrative = ("Case Summary: irregular deposits explained as inheritance, "
                    "verified via estate documents.")
_demo_key_fact = "irregular deposits explained as inheritance, verified via estate documents"
assert score_faithfulness(_demo_narrative, _demo_key_fact) == 1.0
print("Scorers pass smoke test.")

Scorers pass smoke test.


## 4. Run the blind comparison harness

For every synthetic case: generate a narrative from each mock model, score both narratives with the
same scorers, and record the results **under anonymized labels** ("Model A" / "Model B") -- the same
anonymize-before-scoring discipline chapter 2 describes for the human-reviewer side of the
evaluation, applied here to the automated side.

In [5]:
records = []

for case in SYNTHETIC_CASES:
    narrative_a = model_a.generate_narrative(case["context_facts"])
    narrative_b = model_b.generate_narrative(case["context_facts"])

    for label, narrative in [("Model A", narrative_a), ("Model B", narrative_b)]:
        records.append(
            {
                "case_id": case["case_id"],
                "label": label,
                "faithfulness": score_faithfulness(narrative, case["key_fact"]),
                "relevancy": score_relevancy(narrative, case["context_facts"]),
                "narrative_preview": narrative[:70] + ("..." if len(narrative) > 70 else ""),
            }
        )

results_df = pd.DataFrame(records)
results_df

,case_id,label,faithfulness,relevancy,narrative_preview
0,SYN-001,Model A,0.875,1.000,"Case Summary: Customer opened account in 2019, stated occupation: free..."
1,SYN-001,Model B,0.875,1.000,"Case Summary: Customer opened account in 2019, stated occupation: free..."
2,SYN-002,Model A,0.200,0.667,"Case Summary: Customer opened account in 2021, stated occupation: reta..."
3,SYN-002,Model B,0.800,1.000,"Case Summary: Customer opened account in 2021, stated occupation: reta..."
4,SYN-003,Model A,0.000,0.667,"Case Summary: Customer opened account in 2017, stated occupation: soft..."
5,SYN-003,Model B,0.429,1.000,"Case Summary: Customer opened account in 2017, stated occupation: soft..."
6,SYN-004,Model A,0.143,0.667,"Case Summary: Customer opened account in 2020, stated occupation: impo..."
7,SYN-004,Model B,0.714,1.000,"Case Summary: Customer opened account in 2020, stated occupation: impo..."
8,SYN-005,Model A,1.000,1.000,"Case Summary: Customer opened account in 2023, stated occupation: grad..."
9,SYN-005,Model B,1.000,1.000,"Case Summary: Customer opened account in 2023, stated occupation: grad..."


## 5. Side-by-side aggregate score table

This is the anonymized comparison table a researcher (or a risk-review committee) would actually
look at before unblinding -- average faithfulness and relevancy per label, across the full evaluation
set.

In [6]:
summary = (
    results_df.groupby("label")[["faithfulness", "relevancy"]]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)
summary

faithfulness                    relevancy                   
                mean    std    min  max      mean    std    min  max
label                                                               
Model A        0.444  0.459  0.000  1.0       0.8  0.182  0.667  1.0
Model B        0.764  0.214  0.429  1.0       1.0  0.000  1.000  1.0

In [7]:
mean_faithfulness = results_df.groupby("label")["faithfulness"].mean()
diff = round(mean_faithfulness["Model B"] - mean_faithfulness["Model A"], 3)
print(f"Model B faithfulness advantage over Model A: {diff:+.3f} (still blinded at this point)")
assert diff > 0, "Expected Model B to score higher given how the mock generators were built."
print("Comparison harness ran cleanly end-to-end.")

Model B faithfulness advantage over Model A: +0.320 (still blinded at this point)
Comparison harness ran cleanly end-to-end.


## 6. Unblind -- only after scoring is complete

Exactly one unblind step, at the very end, matching chapter 2's rule: never re-check or cherry-pick
after seeing which label maps to which model.

In [8]:
label_to_model = {"Model A": model_a.name, "Model B": model_b.name}
print("Unblinded mapping:")
for label, name in label_to_model.items():
    print(f"  {label} -> {name}")

Unblinded mapping:
  Model A -> internal-backend-alpha
  Model B -> internal-backend-beta


## Recap

This demo reproduced, at small offline scale, the two things chapter 2 argues a defensible model
comparison needs: (1) the *same* scoring instrument applied identically to both models' outputs
(course 4's Ragas-style faithfulness/relevancy pattern), and (2) label anonymization until scoring is
fully complete. A real evaluation would run this same pattern against actual Claude and Azure OpenAI
API responses, at a much larger sample size, with a genuine LLM-judge scorer rather than the
keyword-overlap approximation used here for offline reproducibility -- see chapter 2 for the full
design, including the blind *human* review this notebook doesn't attempt to simulate.